### 1. 离线阶段 - 构建知识库

In [1]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter


# 加载文档
with open("./resources/评估.md", "r", encoding="utf-8") as f:
    md_content = f.read()
# 切分文档
    # 使用结构感知切分器
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "h1"),
            ("##", "h2"),
            ("###", "h3")
        ],
        strip_headers=True
    )
    # 进行切分
    docs: list[Document] = splitter.split_text(md_content)
# 准备向量模型
embaddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",
    dimensions=1024
)
# 准备向量数据库
vector_store = InMemoryVectorStore(embedding=embaddings)
# 调用存储方法
vector_store.add_documents(docs)

['67fae67d-039e-493c-83a9-1fd68c70d817',
 'd418570c-7bff-47a2-80f3-5374c38fded1',
 '3e732ec5-e1dd-4950-9142-45332ee0692a',
 '2c304c80-3dbc-436f-a446-76050d830e37',
 'c54354f9-2d71-4842-ab4c-f7b378733c3a',
 '06e902a5-41f3-42d1-86b6-8bd665bc7ff3',
 '4932093c-1442-48c8-b9a3-1d49cfa6bf00',
 'd3872d44-a420-4739-bad7-0eceb716d3a8',
 'd8a2676d-5d0a-485d-8493-393bf80e3506',
 '885a7d76-2038-41b5-98cb-8700cf2b4719',
 '2efcd614-f8b6-4574-b872-8c845def5c57',
 'a667a447-9bd7-4f20-9891-64c67722dd42',
 '29cc63bd-8e7d-410c-a40b-cf7a47db856a',
 '0e7d1d89-7459-4ca3-bfcf-f41b8e5b8af5',
 '935b9719-f70d-4880-996a-6b5f51185084',
 '543648f0-c179-40a5-a783-a1815f9c4e6f',
 '1e32f9b1-5fe9-4527-a1fe-2d455cbc857d',
 '057d3baf-80f1-491a-928b-6b30f2ebb055',
 '07be2fbe-0c35-4ccd-9e33-8891f486b681',
 '372c7067-55ca-47f1-8b33-e3e998b9a78e',
 'e35a4578-0e31-4baf-94dd-bce73741fa34']

In [4]:
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from app.core.config import settings
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 1. 准备模型
model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={
        "thinking":{
            "type": "disabled"
        }
    }
)
# 2. 准备工具
@tool
def rag_search(query: str, k: int = 4) -> str:
    """
    根据问题， 去向量数据库中检索对应的只是片段
    :param query:用户的问题
    :param k:需要检索的片段数
    :return:得到知识片段
    """
    print("检索工具被调用~")
    # 2.1 检索
    doc_list: list[tuple[Document, float]] = vector_store.similarity_search_with_score(
        query=query, k=k
    )
    # 2.2 增强
    response = ""
    for doc, score in doc_list:
        print(f"分数= {score}")
        print(f"正文： {doc.page_content}")
        response += doc.page_content + "\n\n"
    return response

# 3. 准备系统提示词
system_prompt = """
你是一个教资考试专家，你需要根据提供的上下文回答用户的问题，
不能自己编造答案。答案要简短，直接引用原文。
"""
# 4. 准备Checkpoint
checkpoint = InMemorySaver()
# 5. 定义智能体
_config = {
    "configurable":{
        "thread_id": "1"
    }
}

agent = create_agent(
    model=model,
    tools=[rag_search],
    system_prompt=system_prompt,
    checkpointer=checkpoint
)
# 6. 调用智能体
agent_response = agent.stream_events({
    "messages": [
        HumanMessage(content="教育的目的是什么")
        ]
    },
    version="v3",
    config=_config
)

for msg in agent_response.messages:
    for text in msg.text:
        print(text, end="")

检索工具被调用~
分数= 0.6659331577232094
正文： 教育的本质属性是有目的地培养人的社会活动。  
教育的永恒性。  
教育的历史性。  
教育的相对独立性。  
教育的长期性。  
教育的生产性。  
教育的民族性。
分数= 0.6590694821975029
正文： 广义的教育是指一切有目的地增进人的知识和技能，发展人的智力和体力，影响人的思想品德的社会活动，具有目的性和社会性。广义教育包括社会教育、家庭教育、学校教育。广义的教育是人类社会有史以来就有的教育活动。  
狭义的教育就是指学校教育。  
教育的要素：教育者、受教育者、教育影响（主要是教育内容）。
分数= 0.6505697761043784
正文： 教育与生产劳动和社会生活融合在一起；教育无阶级性、平等性；教育内容简单，有宗教性；教育手段方法单一。
分数= 0.6338490816796786
正文： 政治（经济制度）对教育具有决定作用，是决定教育性质的直接因素。具体表现为：决定了教育的性质和目的；决定了教育的领导权；决定了哪部分社会成员享有受教育的权利；决定了部分的教育内容；决定了教育的管理体制。  
教育对政治发展的作用表现在：教育促进人的政治社会化；教育培养现代政治法律人才；教育促进现代政治民主化。
根据提供的资料，关于“教育的目的是什么”，原文中没有单独、明确的表述。资料中相关的内容是：

- “教育的本质属性是有目的地培养人的社会活动。”
- “政治（经济制度）对教育具有决定作用……具体表现为：决定了教育的性质和目的。”

资料未直接给出教育目的的具体定义或内容。